In [1]:
!pip install datasets evaluate transformers[sentencepiece]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 4.5 MB/s eta 0:00:00


In [2]:
from transformers import AutoTokenizer, DataCollatorWithPadding

In [3]:
import evaluate
import numpy as np

In [4]:
from datasets import load_dataset

raw_datasets = load_dataset("stanfordnlp/sst2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/872 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [5]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [6]:
checkpoint = "google-bert/bert-base-uncased"

In [7]:
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [8]:
def tokenize_function(example):
  return tokenizer(example['sentence'],padding=False,truncation=True)

In [9]:
tokenized_dataset = raw_datasets.map(tokenize_function,batched=True)

Map:   0%|          | 0/67349 [00:00<?, ? examples/s]

Map:   0%|          | 0/872 [00:00<?, ? examples/s]

Map:   0%|          | 0/1821 [00:00<?, ? examples/s]

In [10]:
datacollator = DataCollatorWithPadding(tokenizer=tokenizer)

In [11]:
metric = evaluate.load("accuracy")

In [12]:
from transformers import TrainingArguments

In [13]:
def compute_metrics(eval_preds):
  logits,labels=eval_preds
  predictions = np.argmax(logits,axis=-1)
  return metric.compute(predictions=predictions,references=labels)

In [14]:
training_arguments = TrainingArguments(
    output_dir="./bert-sst2-checkpoints",
    per_device_train_batch_size=16, # During training, one GPU processes 16 samples at one time.
    per_device_eval_batch_size=16, # During evaluation , one GPU processes 16 samples at one time.
    num_train_epochs=3,
    max_steps=-1, # steps dominate epochs , here ignore max steps , do till total steps by epochs , total steps = number of batches * number of epochs
    learning_rate =2e-5,
    lr_scheduler_type ="linear",
    optim="adamw_torch",
    gradient_accumulation_steps=2,
    fp16=True,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end =True,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    save_only_model=True,
    save_total_limit=True,
    logging_dir="./logs",
    logging_steps=50,
    logging_strategy="steps",
    do_eval= True,
    warmup_ratio=0.1,
    weight_decay=0.01,
    dataloader_pin_memory=True,
    remove_unused_columns=True,
    report_to="tensorboard",
    seed=42,






)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
`logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


In [15]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained(checkpoint,num_labels=2)

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [16]:
from transformers import Trainer
trainer = Trainer(
    model=model,
    args=training_arguments,
    data_collator=datacollator,
    train_dataset=tokenized_dataset['train'],
    eval_dataset=tokenized_dataset['validation'],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,

)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.324267,0.234093,0.926606
2,0.220640,0.284549,0.919725
3,0.153278,0.328858,0.923165


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=6315, training_loss=0.30215941728907547, metrics={'train_runtime': 885.1917, 'train_samples_per_second': 228.252, 'train_steps_per_second': 7.134, 'total_flos': 3656008794765660.0, 'train_loss': 0.30215941728907547, 'epoch': 3.0})

In [20]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

In [21]:
# lets look the dataset
print(f"{raw_datasets['train'][1]['sentence']} label={raw_datasets['train'][1]['label']}" )


contains no wit , only labored gags  label=0


In [22]:
from transformers import pipeline
base_classifier = pipeline("text-classification",
                           model="google-bert/bert-base-uncased",
                           tokenizer="google-bert/bert-base-uncased")
print(
    base_classifier("this movie was absolutely fantastic")
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


[{'label': 'LABEL_0', 'score': 0.620048999786377}]


In [23]:
print(
    base_classifier("contains no wit , only labored gags")
)

[{'label': 'LABEL_1', 'score': 0.537886381149292}]


In [25]:

from transformers import pipeline

fine_tuned_classifier = pipeline(
    "text-classification",
    model="./bert-sst2-checkpoints/checkpoint-2105",
    tokenizer="./bert-sst2-checkpoints/checkpoint-2105"
)

print(
    f"This movie was absolutely fantastic! -> "
    f"{fine_tuned_classifier('This movie was absolutely fantastic!')}"
)

print(
    f"This was the worst product I have ever bought. -> "
    f"{fine_tuned_classifier('This was the worst product I have ever bought.')}"
)



Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

This movie was absolutely fantastic! -> [{'label': 'LABEL_1', 'score': 0.9989840388298035}]
This was the worst product I have ever bought. -> [{'label': 'LABEL_0', 'score': 0.9982709884643555}]


In [28]:
print(
    f"I love this phone -> "
    f"{fine_tuned_classifier('I love this phone')}"
)

I love this phone -> [{'label': 'LABEL_1', 'score': 0.9971000552177429}]


In [29]:
## model already trained , but we want model to show negative for 0 and 1 for positive  so ,
model.config.id2label = { 0: "NEGATIVE", 1: "POSITIVE" }
model.config.label2id = { "NEGATIVE": 0, "POSITIVE": 1 }

In [30]:
trainer.save_model("./final-sst2-model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [32]:

from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./final-sst2-model",
    tokenizer="./final-sst2-model"
)




Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [34]:
result = classifier( "This AI project is absolutely amazing!" )
print(result)

[{'label': 'POSITIVE', 'score': 0.9989537000656128}]


In [35]:

sentences = [

    "I love this movie",

    "This was the worst experience ever",

    "The product quality is amazing",

    "I regret buying this phone",

    "The movie was okay"
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):

    print(f"Sentence: {sentence}")

    print(f"Prediction: {result}")

    print("-" * 50)



Sentence: I love this movie
Prediction: {'label': 'POSITIVE', 'score': 0.9986357092857361}
--------------------------------------------------
Sentence: This was the worst experience ever
Prediction: {'label': 'NEGATIVE', 'score': 0.9980185031890869}
--------------------------------------------------
Sentence: The product quality is amazing
Prediction: {'label': 'POSITIVE', 'score': 0.999053418636322}
--------------------------------------------------
Sentence: I regret buying this phone
Prediction: {'label': 'NEGATIVE', 'score': 0.9919916987419128}
--------------------------------------------------
Sentence: The movie was okay
Prediction: {'label': 'POSITIVE', 'score': 0.9944403767585754}
--------------------------------------------------


In [36]:
sentences = [

    "The movie was not bad",

    "I expected much better",

    "It was decent but too long",

    "The acting was terrible but visuals were amazing",

    "I don't think I disliked it",

    "The product is fine",

    "The ending ruined the entire film",

    "Not great, not terrible",

    "This is probably one of the movies ever made",

    "It had potential but failed badly"
]

results = classifier(sentences)

for sentence, result in zip(sentences, results):

    print(f"Sentence: {sentence}")

    print(f"Prediction: {result}")

    print("-" * 50)


Sentence: The movie was not bad
Prediction: {'label': 'POSITIVE', 'score': 0.9943391680717468}
--------------------------------------------------
Sentence: I expected much better
Prediction: {'label': 'NEGATIVE', 'score': 0.8731109499931335}
--------------------------------------------------
Sentence: It was decent but too long
Prediction: {'label': 'NEGATIVE', 'score': 0.5144718289375305}
--------------------------------------------------
Sentence: The acting was terrible but visuals were amazing
Prediction: {'label': 'POSITIVE', 'score': 0.9969315528869629}
--------------------------------------------------
Sentence: I don't think I disliked it
Prediction: {'label': 'NEGATIVE', 'score': 0.5035524368286133}
--------------------------------------------------
Sentence: The product is fine
Prediction: {'label': 'POSITIVE', 'score': 0.9989124536514282}
--------------------------------------------------
Sentence: The ending ruined the entire film
Prediction: {'label': 'NEGATIVE', 'score': 

In [37]:
from huggingface_hub import notebook_login

notebook_login()

In [38]:
model.push_to_hub(
    "bert-base-uncased-sst2-finetuned"
)

tokenizer.push_to_hub(
    "bert-base-uncased-sst2-finetuned"
)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...__en9pg/model.safetensors:   0%|          | 14.2kB /  438MB            

README.md: 0.00B [00:00, ?B/s]

CommitInfo(commit_url='https://huggingface.co/baiju0110/bert-base-uncased-sst2-finetuned/commit/5038d4c64004ae84ed8e31452dc40720203645c6', commit_message='Upload tokenizer', commit_description='', oid='5038d4c64004ae84ed8e31452dc40720203645c6', pr_url=None, repo_url=RepoUrl('https://huggingface.co/baiju0110/bert-base-uncased-sst2-finetuned', endpoint='https://huggingface.co', repo_type='model', repo_id='baiju0110/bert-base-uncased-sst2-finetuned'), pr_revision=None, pr_num=None)